# 02 Image Stretching

Raw astrophotography data is very dark, stretching redistributes pixel values to reveal hidden structure.

**Methods covered:** ZScale, Asinh + Percentile

In [ ]:
from astropy.io import fits
from astropy.visualization import ZScaleInterval, AsinhStretch, PercentileInterval
import numpy as np
import matplotlib.pyplot as plt

fits_file = "your_image.fits"  # Change this to your file path

with fits.open(fits_file) as hdul:
    data = hdul[0].data

rgb = np.transpose(data, (1, 2, 0))

## Method 1 ZScale

Used in professional astronomy. Intelligently determines the display range based on data distribution.

### How it works
`ZScaleInterval` scans the pixel values and finds a display range that shows the most detail.
`np.clip` ensures all values stay between 0 and 1.
The loop processes each RGB channel separately.

In [ ]:
interval = ZScaleInterval()
rgb_z = np.zeros_like(rgb)

for i in range(3):
    vmin, vmax = interval.get_limits(rgb[:,:,i])
    rgb_z[:,:,i] = np.clip((rgb[:,:,i] - vmin) / (vmax - vmin), 0, 1)

plt.figure(figsize=(10, 6))
plt.imshow(rgb_z, origin='lower')
plt.title('ZScale Stretch')
plt.axis('off')
plt.show()

## Method 2 Asinh + Percentile (Recommended for Nebulae)

Best for images with a bright core and faint outer regions.

### How it works
- `PercentileInterval(99.5)` ignores the top 0.5% of pixel values to avoid overexposure
- `AsinhStretch(a=0.01)` applies a non-linear stretch — lower `a` = stronger stretch
- The loop processes each RGB channel separately

Adjust `a` to control the result:
- `a=0.001` — very strong stretch
- `a=0.01` — balanced (recommended)
- `a=0.1` — gentle stretch

In [ ]:
interval = PercentileInterval(99.5)
stretch  = AsinhStretch(a=0.01)

rgb_new = np.zeros_like(rgb)
for i in range(3):
    vmin, vmax     = interval.get_limits(rgb[:,:,i])
    normalized     = np.clip((rgb[:,:,i] - vmin) / (vmax - vmin), 0, 1)
    rgb_new[:,:,i] = stretch(normalized)

plt.figure(figsize=(10, 6))
plt.imshow(rgb_new, origin='lower')
plt.title('Asinh Stretch')
plt.axis('off')
plt.show()

## Before vs After

Side-by-side comparison of the raw image and the stretched version.